In [2]:
# pip install gymnasium torch numpy matplotlib

import random
import collections
import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


ENV_NAME = "CartPole-v1"

learning_rate = 0.0005
gamma = 0.98
buffer_limit = 50000
batch_size = 64

num_episodes = 4000
target_update_interval = 20

epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 0.995

train_start_size = 2000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


class ReplayBuffer:
    def __init__(self):
        self.buffer = collections.deque(maxlen=buffer_limit)

    def put(self, transition):
        self.buffer.append(transition)

    def sample(self, n):
        mini_batch = random.sample(self.buffer, n)

        s_lst, a_lst, r_lst, s_prime_lst, done_lst = [], [], [], [], []

        for s, a, r, s_prime, done in mini_batch:
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r])
            s_prime_lst.append(s_prime)
            done_lst.append([done])

        s = torch.tensor(np.array(s_lst), dtype=torch.float32).to(device)
        a = torch.tensor(a_lst, dtype=torch.long).to(device)
        r = torch.tensor(r_lst, dtype=torch.float32).to(device)
        s_prime = torch.tensor(np.array(s_prime_lst), dtype=torch.float32).to(device)
        done = torch.tensor(done_lst, dtype=torch.float32).to(device)

        return s, a, r, s_prime, done

    def size(self):
        return len(self.buffer)


class QNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNet, self).__init__()

        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        q = self.fc3(x)
        return q

    def sample_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randint(0, 1)

        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            q_values = self.forward(state)

        return q_values.argmax().item()


def train(q_net, target_q_net, memory, optimizer):
    q_net.train()

    for _ in range(10):
        s, a, r, s_prime, done = memory.sample(batch_size)

        q_values = q_net(s)
        q_a = q_values.gather(1, a)

        with torch.no_grad():
            max_q_prime = target_q_net(s_prime).max(1)[0].unsqueeze(1)
            target = r + gamma * max_q_prime * (1 - done)

        loss = F.smooth_l1_loss(q_a, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def main():
    env = gym.make(ENV_NAME)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    q_net = QNet(state_dim, action_dim).to(device)
    target_q_net = QNet(state_dim, action_dim).to(device)
    target_q_net.load_state_dict(q_net.state_dict())

    memory = ReplayBuffer()
    optimizer = optim.Adam(q_net.parameters(), lr=learning_rate)

    epsilon = epsilon_start
    reward_history = []

    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = q_net.sample_action(state, epsilon)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            memory.put((state, action, reward, next_state, float(done)))

            state = next_state
            total_reward += reward

            if memory.size() > train_start_size:
                train(q_net, target_q_net, memory, optimizer)

        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        if episode % target_update_interval == 0:
            target_q_net.load_state_dict(q_net.state_dict())

        reward_history.append(total_reward)

        if episode % 10 == 0:
            avg_reward = np.mean(reward_history[-10:])
            print(
                f"Episode: {episode:4d} | "
                f"Reward: {total_reward:6.1f} | "
                f"Avg Reward: {avg_reward:6.1f} | "
                f"Epsilon: {epsilon:.3f}"
            )

        if len(reward_history) >= 20 and np.mean(reward_history[-20:]) >= 475:
            print("Solved!")
            break

    env.close()

    torch.save(q_net.state_dict(), "dqn_cartpole.pth")
    print("Model saved: dqn_cartpole.pth")


if __name__ == "__main__":
    main()

Device: cuda
Episode:    0 | Reward:   12.0 | Avg Reward:   12.0 | Epsilon: 0.995
Episode:   10 | Reward:   16.0 | Avg Reward:   23.4 | Epsilon: 0.946
Episode:   20 | Reward:   21.0 | Avg Reward:   21.9 | Epsilon: 0.900
Episode:   30 | Reward:   31.0 | Avg Reward:   23.1 | Epsilon: 0.856
Episode:   40 | Reward:   12.0 | Avg Reward:   18.9 | Epsilon: 0.814
Episode:   50 | Reward:   12.0 | Avg Reward:   17.5 | Epsilon: 0.774
Episode:   60 | Reward:   12.0 | Avg Reward:   18.7 | Epsilon: 0.737
Episode:   70 | Reward:   27.0 | Avg Reward:   21.0 | Epsilon: 0.701
Episode:   80 | Reward:   13.0 | Avg Reward:   19.1 | Epsilon: 0.666
Episode:   90 | Reward:   17.0 | Avg Reward:   14.3 | Epsilon: 0.634
Episode:  100 | Reward:   18.0 | Avg Reward:   15.6 | Epsilon: 0.603
Episode:  110 | Reward:   11.0 | Avg Reward:   27.6 | Epsilon: 0.573
Episode:  120 | Reward:   32.0 | Avg Reward:   48.0 | Epsilon: 0.545
Episode:  130 | Reward:   25.0 | Avg Reward:   27.7 | Epsilon: 0.519
Episode:  140 | Rewar